# SQL Worksheet — Week1

Use the following tables from the Riva Data Platform:

- `rivadataplatform.dataproduct.dim_batch`
- `rivadataplatform.dataproduct.dim_class`
- `rivadataplatform.dataproduct.fact_attendance`
- `rivadataplatform.dataproduct.dim_student`
- `rivadataplatform.dataproduct.dim_date`

**Instructions**
- Write SQL for each question.
- Do not modify the source data.
- Use clear aliases where JOINs are involved.
- Unless a question specifically asks for a particular column, select only the columns needed to answer it.


## Tables / Relationships

Useful keys:
- `dim_student.student_key` ↔ `fact_attendance.student_key`
- `dim_class.class_key` ↔ `fact_attendance.class_key`
- `dim_batch.batch_key` ↔ `fact_attendance.batch_key`
- `dim_class.batch_id` ↔ `dim_batch.batch_id`

## Question 1 — Student Attendance Profile
Build the query in these steps:

**1.1 — Select student columns**
From `dim_student`, select `student_id` and `student_name`.

**1.2 — Add attendance data**
Join `fact_attendance` with a `LEFT JOIN` using `student_key`, then add the attendance record identifier.

**1.3 — Add class and batch data**
Join `dim_class` using `class_key` and `dim_batch` using `batch_key`. Add the batch name.

**1.4 — Group and aggregate**
Group by student and batch name. Count attendance records and count missing phone numbers using `NULLIF` and `CASE`.

**1.5 — Sort the result**
Order by attendance records descending, then student name.

Return one row per student and batch, including students with no attendance records.

In [ ]:
SELECT
    s.student_id,
    s.student_name,
    COALESCE(b.batch_name, 'No batch') AS batch_name,
    COUNT(f.attendance_id) AS attendance_records,
    SUM(CASE WHEN NULLIF(s.phone_no, '') IS NULL THEN 1 ELSE 0 END) AS missing_phone_rows
FROM rivadataplatform.dataproduct.dim_student AS s
LEFT JOIN rivadataplatform.dataproduct.fact_attendance AS f
    ON f.student_key = s.student_key
LEFT JOIN rivadataplatform.dataproduct.dim_class AS c
    ON c.class_key = f.class_key
LEFT JOIN rivadataplatform.dataproduct.dim_batch AS b
    ON b.batch_key = f.batch_key
GROUP BY s.student_id, s.student_name, b.batch_name
ORDER BY attendance_records DESC, s.student_name;

## Question 2 — Missing Profile Data by City
Build the query in these steps:

**2.1 — Select student location**
From `dim_student`, select the city and replace null city values with `Unknown city`.

**2.2 — Join attendance**
Join `fact_attendance` to the students using `student_key`, then add the attendance record identifier.

**2.3 — Add class information**
Join `dim_class` using `class_key` and add the class topic, replacing null topics with `Topic not assigned`.

**2.4 — Group and aggregate**
Group by the null-safe city and topic. Count distinct students, attendance records, and missing phone values.

**2.5 — Filter and sort**
Keep only groups with at least one missing phone number. Order by missing phone count descending, then city and topic.

In [ ]:
SELECT
    COALESCE(s.city, 'Unknown city') AS city,
    COALESCE(c.topic, 'Topic not assigned') AS topic,
    COUNT(DISTINCT s.student_key) AS distinct_students,
    COUNT(f.attendance_id) AS attendance_records,
    SUM(CASE WHEN NULLIF(s.phone_no, '') IS NULL THEN 1 ELSE 0 END) AS missing_phone_count
FROM rivadataplatform.dataproduct.dim_student AS s
LEFT JOIN rivadataplatform.dataproduct.fact_attendance AS f
    ON f.student_key = s.student_key
LEFT JOIN rivadataplatform.dataproduct.dim_class AS c
    ON c.class_key = f.class_key
GROUP BY COALESCE(s.city, 'Unknown city'), COALESCE(c.topic, 'Topic not assigned')
HAVING SUM(CASE WHEN NULLIF(s.phone_no, '') IS NULL THEN 1 ELSE 0 END) > 0
ORDER BY missing_phone_count DESC, city, topic;

## Question 3 — Distinct Class Calendar
Build the query in these steps:

**3.1 — Select class date columns**
From `dim_class`, select the class date and class key.

**3.2 — Join attendance**
Join `fact_attendance` using `class_key`, then add the attendance record identifier and `date_key`.

**3.3 — Add batch and calendar details**
Join `dim_batch` using the class batch relationship and join `dim_date` using `date_key`. Add batch name, topic, and day name.

**3.4 — Group and count**
Return one row per distinct class date and count its attendance records.

**3.5 — Sort chronologically**
Order the result by class date.

In [ ]:
SELECT DISTINCT
    c.class_date,
    b.batch_name,
    COALESCE(c.topic, 'Topic not assigned') AS topic,
    d.day_name,
    COUNT(f.attendance_id) OVER (PARTITION BY c.class_key) AS attendance_records
FROM rivadataplatform.dataproduct.dim_class AS c
JOIN rivadataplatform.dataproduct.fact_attendance AS f
    ON f.class_key = c.class_key
JOIN rivadataplatform.dataproduct.dim_batch AS b
    ON b.batch_id = c.batch_id
LEFT JOIN rivadataplatform.dataproduct.dim_date AS d
    ON d.date_key = f.date_key
ORDER BY c.class_date;

## Question 4 — Student and Class Detail
Build the query in these steps:

**4.1 — Select student columns**
From `dim_student`, select the student name and city, replacing a null city with `Unknown city`.

**4.2 — Join attendance**
Start from `fact_attendance` and join `dim_student` using `student_key`. Select the attendance status.

**4.3 — Add class, batch, and date details**
Join `dim_class`, `dim_batch`, and `dim_date` using their keys. Add class date, day name, batch name, and topic with readable null labels.

**4.4 — Filter valid records**
Keep only rows where attendance status is not null or blank.

**4.5 — Sort the detail**
Order by class date, then student name.

In [ ]:
SELECT
    s.student_name,
    COALESCE(s.city, 'Unknown city') AS city,
    COALESCE(b.batch_name, 'No batch') AS batch_name,
    c.class_date,
    COALESCE(d.day_name, c.class_day) AS day_name,
    COALESCE(c.topic, 'Topic not assigned') AS topic,
    f.attendance_status
FROM rivadataplatform.dataproduct.fact_attendance AS f
JOIN rivadataplatform.dataproduct.dim_student AS s
    ON s.student_key = f.student_key
JOIN rivadataplatform.dataproduct.dim_class AS c
    ON c.class_key = f.class_key
LEFT JOIN rivadataplatform.dataproduct.dim_batch AS b
    ON b.batch_key = f.batch_key
LEFT JOIN rivadataplatform.dataproduct.dim_date AS d
    ON d.date_key = f.date_key
WHERE NULLIF(f.attendance_status, '') IS NOT NULL
ORDER BY c.class_date, s.student_name;

## Question 5 — Attendance Status Summary
Build the query in these steps:

**5.1 — Select grouping columns**
From attendance, select the batch name and class topic, replacing null values with `No batch` and `Topic not assigned`.

**5.2 — Join dimensions**
Join `dim_class` using `class_key` and `dim_batch` using `batch_key`.

**5.3 — Count attendance**
Count total attendance records and distinct students.

**5.4 — Count each status**
Use conditional aggregation to count `Present`, `Late`, and `Absent` records.

**5.5 — Keep and sort groups**
Keep groups with at least one attendance record and order by batch name, then topic.

In [ ]:
SELECT
    COALESCE(b.batch_name, 'No batch') AS batch_name,
    COALESCE(c.topic, 'Topic not assigned') AS topic,
    COUNT(f.attendance_id) AS attendance_records,
    COUNT(DISTINCT f.student_key) AS distinct_students,
    SUM(CASE WHEN f.attendance_status = 'Present' THEN 1 ELSE 0 END) AS present_count,
    SUM(CASE WHEN f.attendance_status = 'Late' THEN 1 ELSE 0 END) AS late_count,
    SUM(CASE WHEN f.attendance_status = 'Absent' THEN 1 ELSE 0 END) AS absent_count
FROM rivadataplatform.dataproduct.fact_attendance AS f
LEFT JOIN rivadataplatform.dataproduct.dim_class AS c
    ON c.class_key = f.class_key
LEFT JOIN rivadataplatform.dataproduct.dim_batch AS b
    ON b.batch_key = f.batch_key
GROUP BY COALESCE(b.batch_name, 'No batch'), COALESCE(c.topic, 'Topic not assigned')
HAVING COUNT(f.attendance_id) > 0
ORDER BY batch_name, topic;

## Question 6 — Students Requiring Follow-up
Build the query in these steps:

**6.1 — Select student columns**
From `dim_student`, select `student_id` and `student_name`.

**6.2 — Join attendance and dimensions**
Use `LEFT JOIN` to add attendance, class, and batch data through their keys.

**6.3 — Calculate student metrics**
Group by student and calculate total attendance records, issue records, and the most recent class date.

**6.4 — Identify issues**
Count rows whose status is `Late` or `Absent`, and keep only students with at least one such issue.

**6.5 — Sort for follow-up**
Order by issue count descending, then student name.

Return each student once.

In [ ]:
SELECT
    s.student_id,
    s.student_name,
    COALESCE(MAX(b.batch_name), 'No batch') AS batch_name,
    COUNT(f.attendance_id) AS total_attendance_records,
    SUM(CASE WHEN f.attendance_status IN ('Late', 'Absent') THEN 1 ELSE 0 END) AS issue_count,
    MAX(c.class_date) AS most_recent_class_date
FROM rivadataplatform.dataproduct.dim_student AS s
LEFT JOIN rivadataplatform.dataproduct.fact_attendance AS f
    ON f.student_key = s.student_key
LEFT JOIN rivadataplatform.dataproduct.dim_class AS c
    ON c.class_key = f.class_key
LEFT JOIN rivadataplatform.dataproduct.dim_batch AS b
    ON b.batch_key = f.batch_key
GROUP BY s.student_id, s.student_name
HAVING SUM(CASE WHEN f.attendance_status IN ('Late', 'Absent') THEN 1 ELSE 0 END) > 0
ORDER BY issue_count DESC, s.student_name;